# SHIVA-LLM · Objective 02 · Stage 4A-B
## Simulator robustness and reward-dominance audit
**Runtime:** Google Colab CPU, Python 3. **Input:** `objective02_stage4a_simulator.zip`.

This audit tests predictive calibration, learner-level uncertainty, and **assumption-based** reward dominance. It does **not** estimate causal instructional effects or validate sequential learning dynamics. `p_observed` is the Stage 4A response-model prediction, **not the original HyperKT probability**; the latter is not exported in the Stage 4A predictions file.

In [ ]:
import io, json, zipfile, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.special import expit, logit
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
SEED=42; rng=np.random.default_rng(SEED)
OUT=Path('/content/stage4ab_outputs') if Path('/content').exists() else Path('stage4ab_outputs')
OUT.mkdir(parents=True,exist_ok=True)
try:
 from google.colab import files
 uploaded=files.upload()
 source=Path(next(k for k in uploaded if k.lower().endswith('.zip')))
except ImportError:
 source=Path('objective02_stage4a_simulator.zip')
assert source.exists(), f'Missing input: {source}'
with zipfile.ZipFile(source) as z:
 print('Input files:',z.namelist())
 required={'response_model_predictions.csv','response_model_metrics.csv','simulation_sensitivity.csv','simulator_spec.json'}
 assert required.issubset(set(z.namelist())),f'Missing: {required-set(z.namelist())}'
 df=pd.read_csv(io.BytesIO(z.read('response_model_predictions.csv')))
 old=pd.read_csv(io.BytesIO(z.read('simulation_sensitivity.csv')))
 spec=json.loads(z.read('simulator_spec.json'))
print('Rows:',len(df),'Split counts:',df.split.value_counts().to_dict())

## 1 · Integrity and empirical calibration
We evaluate **held-out predictions** without fitting a new model. The probability-bin calibration groups are descriptive; readiness bins use `p_observed` as a **proxy** and should not be interpreted as independently measured learner readiness.

In [ ]:
assert {'split','user_id','order_id','observed_correct','p_observed'}.issubset(df.columns)
assert df.split.isin(['train','val','test']).all()
assert df.p_observed.between(0,1).all() and df.observed_correct.isin([0,1]).all()
assert not df.duplicated(['user_id','order_id']).any()
parts={s:set(df.loc[df.split==s,'user_id']) for s in ['train','val','test']}
assert all(not(parts[a]&parts[b]) for a,b in [('train','val'),('train','test'),('val','test')])
assert spec.get('counterfactual_effect_identified') is False
assert spec.get('longitudinal_transition_model_validated') is False
print('Integrity checks passed; learner partitions are disjoint.')

def metrics(d):
 y=d.observed_correct.to_numpy(); p=np.clip(d.p_observed.to_numpy(),1e-7,1-1e-7)
 return dict(n=len(d),learners=d.user_id.nunique(),observed_rate=y.mean(),predicted_rate=p.mean(),
             calibration_in_the_large=p.mean()-y.mean(),brier=brier_score_loss(y,p),
             log_loss=log_loss(y,p,labels=[0,1]),auc=roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan)
summary=pd.DataFrame([{'split':s,**metrics(df[df.split==s])} for s in ['train','val','test']])
summary.to_csv(OUT/'heldout_metrics.csv',index=False);display(summary)

# Fixed bins permit comparison between splits; empty bins are omitted.
bins=np.linspace(0,1,11)
df['probability_bin']=pd.cut(df.p_observed,bins,include_lowest=True)
rows=[]
for (split,bin_),g in df.groupby(['split','probability_bin'],observed=True):
 rows.append({'split':split,'bin':str(bin_),'n':len(g),'predicted':g.p_observed.mean(),'observed':g.observed_correct.mean(),'gap':g.p_observed.mean()-g.observed_correct.mean()})
cal=pd.DataFrame(rows);cal.to_csv(OUT/'calibration_by_probability_bin.csv',index=False)
fig,ax=plt.subplots(figsize=(6,5));ax.plot([0,1],[0,1],'k--',label='Perfect calibration')
for s in ['val','test']:
 d=cal[cal.split==s];ax.plot(d.predicted,d.observed,'o-',label=s)
ax.set(xlabel='Mean predicted probability',ylabel='Observed correctness',title='Held-out reliability by probability bin');ax.legend();fig.tight_layout();fig.savefig(OUT/'heldout_reliability.png',dpi=160);plt.show()

## 2 · Learner-cluster bootstrap
Bootstrap **learners**, not individual interactions, to account for repeated observations. Confidence intervals describe uncertainty in held-out predictive metrics, not policy-effect estimates.

In [ ]:
def learner_bootstrap(d,B=300,seed=42):
 r=np.random.default_rng(seed);groups=[g[['observed_correct','p_observed']].to_numpy() for _,g in d.groupby('user_id',sort=False)]
 m=len(groups); vals=[]
 for _ in range(B):
  idx=r.integers(0,m,size=m); a=np.concatenate([groups[i] for i in idx],axis=0)
  y=a[:,0];p=np.clip(a[:,1],1e-7,1-1e-7)
  vals.append([np.mean((y-p)**2),np.mean(p-y),roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan])
 return np.nanpercentile(np.asarray(vals),[2.5,50,97.5],axis=0)
ci=[]
for s in ['val','test']:
 a=learner_bootstrap(df[df.split==s],B=300,seed=SEED)
 for j,name in enumerate(['brier','calibration_in_the_large','auc']):ci.append({'split':s,'metric':name,'ci_2_5':a[0,j],'median':a[1,j],'ci_97_5':a[2,j]})
ci=pd.DataFrame(ci);ci.to_csv(OUT/'learner_bootstrap_ci.csv',index=False);display(ci)

## 3 · Readiness-stratified diagnostic
Here **readiness = response-model predicted correctness**, used only as a diagnostic proxy. Low, middle and high groups use fixed probability thresholds and are evaluated on held-out data.

In [ ]:
df['readiness_proxy']=pd.cut(df.p_observed,[-0.001,0.4,0.7,1.001],labels=['low (<0.4)','middle (0.4–0.7)','high (>0.7)'])
groups=[]
for (s,r),g in df[df.split.isin(['val','test'])].groupby(['split','readiness_proxy'],observed=True):
 groups.append({'split':s,'readiness_proxy':str(r),**metrics(g)})
groups=pd.DataFrame(groups);groups.to_csv(OUT/'readiness_calibration.csv',index=False);display(groups)

## 4 · Reproduce Stage 4A simulator and audit action dominance
We reproduce the **assumed** one-step reward: 0.60 × correctness + 0.20 × efficiency + 0.20 × suitability. Assumed difficulty costs are [0.15, 0.35, 0.60], and suitability rewards proximity to correctness probability 0.65. No actual action effects are inferred from the data.

In [ ]:
scenarios=spec['action_effect_log_odds_scenarios']
costs=np.array([spec['assumed_action_costs'][x] for x in ['easy','medium','hard']],dtype=float)
base=df.loc[df.split=='test','p_observed'].to_numpy()
def rewards(base,effect,weights=(0.6,0.2,0.2),target=0.65):
 p=expit(logit(np.clip(base,1e-5,1-1e-5))[:,None]+effect*(1-np.arange(3))[None,:])
 suitability=np.clip(1-np.abs(p-target)/target,0,1)
 return weights[0]*p+weights[1]*(1-costs[None,:])+weights[2]*suitability
res=[]
for name,effect in scenarios.items():
 R=rewards(base,effect);best=np.argmax(R,axis=1)
 for a,label in enumerate(['easy','medium','hard']):
  res.append({'scenario':name,'action':label,'mean_reward':R[:,a].mean(),'optimal_fraction':(best==a).mean(),
              'mean_regret_vs_oracle':np.mean(np.max(R,axis=1)-R[:,a])})
 res.append({'scenario':name,'action':'per-context oracle','mean_reward':np.max(R,axis=1).mean(),'optimal_fraction':1.0,'mean_regret_vs_oracle':0.0})
dom=pd.DataFrame(res);dom.to_csv(OUT/'action_dominance.csv',index=False);display(dom)
print('Note: oracle is optimal only under the declared simulation assumptions.')

# Verify Stage 4A fixed-policy means are reproduced (numerical tolerance).
for name in scenarios:
 for a,label in enumerate(['fixed_easy','fixed_medium','fixed_hard']):
  oldrow=old[(old.scenario==name)&(old.policy==label)]
  if len(oldrow):
   expected=float(oldrow.iloc[0].mean_simulated_expected_reward)
   actual=float(dom[(dom.scenario==name)&(dom.action==['easy','medium','hard'][a])].iloc[0].mean_reward)
   assert abs(actual-expected)<2e-5,(name,label,actual,expected)
print('Stage 4A fixed-policy reward reproduction passed.')

## 5 · Reward weights and action-effect sensitivity
A grid of plausible **assumptions**, not empirically identified parameters. If one action wins throughout, a contextual bandit may have little to learn under this simulator. We do not tune assumptions to force an adaptive policy.

In [ ]:
weight_sets={'original':(0.6,0.2,0.2),'correctness_heavy':(0.8,0.1,0.1),
 'suitability_heavy':(0.35,0.1,0.55),'efficiency_heavy':(0.35,0.5,0.15),
 'balanced':(1/3,1/3,1/3)}
records=[]
for scen,effect in {'zero':0.0,**scenarios,'reverse_moderate':-0.4}.items():
 for wname,w in weight_sets.items():
  for target in [0.5,0.65,0.8]:
   R=rewards(base,effect,w,target);best=R.argmax(axis=1);counts=np.bincount(best,minlength=3)/len(best)
   records.append({'scenario':scen,'effect':effect,'weights':wname,'target':target,
                   'optimal_easy_fraction':counts[0],'optimal_medium_fraction':counts[1],'optimal_hard_fraction':counts[2],
                   'mean_oracle_reward':np.max(R,axis=1).mean(),
                   'mean_fixed_easy_reward':R[:,0].mean(),'oracle_minus_easy':np.mean(np.max(R,axis=1)-R[:,0])})
sens=pd.DataFrame(records);sens.to_csv(OUT/'reward_weight_sensitivity.csv',index=False);display(sens)
print('Scenarios with >95% easy dominance:',int((sens.optimal_easy_fraction>0.95).sum()),'of',len(sens))

## 6 · Decision report and export
A conservative gate is produced. **No numerical calibration threshold can establish causal validity**; the report separately records that instructional action effects and longitudinal transitions remain unvalidated.

In [ ]:
test=summary[summary.split=='test'].iloc[0]
original=sens[(sens.scenario=='moderate')&(sens.weights=='original')&(sens.target==0.65)].iloc[0]
report={
 'stage':'4A-B','input_zip':source.name,'n_test':int(test.n),'test_brier':float(test.brier),
 'test_auc':float(test.auc),'test_calibration_in_the_large':float(test.calibration_in_the_large),
 'moderate_original_easy_optimal_fraction':float(original.optimal_easy_fraction),
 'moderate_original_oracle_minus_easy':float(original.oracle_minus_easy),
 'predictive_metrics_are_observational':True,'action_effects_are_assumed':True,
 'causal_action_effects_identified':False,'sequential_transition_model_validated':False,
 'real_world_policy_gain_established':False,'ready_for_causal_offline_rl':False,
 'next_recommendation':'Do not claim causal policy improvement. Review reward dominance; if proceeding, use assumption-labeled one-step contextual-bandit benchmarks only.'}
(OUT/'audit_report.json').write_text(json.dumps(report,indent=2))
(OUT/'README.txt').write_text('SHIVA-LLM Objective 02 Stage 4A-B. Observational predictive calibration and assumed one-step reward robustness only. No authenticated interventions, causal action effects, or validated longitudinal transitions.\n')
print(json.dumps(report,indent=2))
archive=Path('/content/objective02_stage4ab_robustness_audit.zip') if Path('/content').exists() else Path('objective02_stage4ab_robustness_audit.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
 for f in sorted(OUT.iterdir()):z.write(f,f.name)
print('Created:',archive,'bytes:',archive.stat().st_size)
try:
 from google.colab import files
 files.download(str(archive))
except ImportError:pass